# 03 - Feature Engineering
Create numerical text features, TF-IDF features, and inspect informative financial terms.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from src.data_loader import load_financial_phrasebank
from src.preprocessing import preprocess_dataframe
from src.feature_engineering import add_text_features
df = preprocess_dataframe(load_financial_phrasebank())
df = add_text_features(df)
df[['sentiment','word_count','char_count']].head()

In [ ]:
print(df.groupby('sentiment')[['word_count','char_count']].mean().round(2))
df.groupby('sentiment')['word_count'].mean().plot(kind='bar', title='Average Text Length by Sentiment')
plt.ylabel('Average Word Count')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['sentiment'], test_size=0.2, random_state=42, stratify=df['sentiment']
)

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1,2), min_df=2, sublinear_tf=True)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)
print('Training matrix:', X_train_tfidf.shape)
print('Testing matrix:', X_test_tfidf.shape)

In [ ]:
feature_names = vectorizer.get_feature_names_out()
tfidf_means = X_train_tfidf.mean(axis=0).A1
top_features = pd.DataFrame({'term': feature_names, 'mean_tfidf': tfidf_means}).sort_values('mean_tfidf', ascending=False).head(25)
top_features

In [ ]:
top_features.set_index('term')['mean_tfidf'].sort_values().plot(kind='barh', title='Top TF-IDF Features')
plt.xlabel('Mean TF-IDF')
plt.tight_layout()
plt.show()